# Análise Exploratória de `dados.parquet` com PySpark

Este notebook reproduz a mesma análise descritiva do relatório `EDA.md`, agora com **PySpark**. Todos os cálculos (contagens, estatísticas, quantis, correlações, VIF, AUC, testes, densidades e agregações para os gráficos) são feitos no Spark. Para o driver vão apenas os **resultados agregados**, usados pelo `matplotlib` para desenhar. A exceção é o gráfico de dispersão, que usa uma amostra de 2.500 linhas.

**Variável resposta:** `ALVO` (binária).

**Requisitos:**
- `pip install pyspark` (testado com PySpark 4.2)
- **Java 17+** (`JAVA_HOME` configurado)
- `numpy`, `pandas`, `matplotlib`, `scipy`. O `scipy` é usado só para obter p-valores das distribuições teóricas (normal), nunca para processar os dados.

| Seção | Conteúdo |
|---|---|
| 0 | Configuração e leitura |
| 1 | Estrutura e qualidade da base |
| 2 | Variável resposta e vazamento |
| 3 | Estatísticas descritivas, histogramas e outliers |
| 4 | Matrizes de correlação (Pearson e Spearman) e VIF |
| 5 | Relação das variáveis numéricas com o ALVO (AUC, Mann-Whitney, KDE, boxplots, dispersão) |
| 6 | Variáveis categóricas (qui-quadrado, V de Cramér, taxas por grupo) |
| 7 | Baselines |

## 0. Configuração

In [0]:
import os, sys
# Em Windows, o Spark precisa saber qual Python usar nos workers.
os.environ.setdefault("PYSPARK_PYTHON", sys.executable)
os.environ.setdefault("PYSPARK_DRIVER_PYTHON", sys.executable)
# Se o Java não estiver no PATH, descomente e ajuste:
# os.environ["JAVA_HOME"] = r"C:\caminho\para\jdk-17"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from scipy import stats  # apenas distribuições teóricas (p-valores)

from pyspark.sql import SparkSession, Window, functions as F
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.stat import Correlation, ChiSquareTest
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.mllib.stat import KernelDensity

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

In [0]:
spark = (SparkSession.builder
         .appName("EDA")
         .master("local[*]")
         .config("spark.sql.shuffle.partitions", "8")
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
spark.version

In [0]:
CAMINHO = r"C:/Users/Administrador/Downloads/dados.parquet"
df = spark.read.parquet(CAMINHO).cache()
N = df.count()
print(f"{N} linhas x {len(df.columns)} colunas")
df.printSchema()

In [0]:
# Paleta e estilo dos gráficos
BLUE, ORANGE, INK, INK2, MUTED, SURF = "#2a78d6", "#eb6834", "#0b0b0b", "#52514e", "#898781", "#fcfcfb"
COR = {0: ORANGE, 1: BLUE}
plt.rcParams.update({
    "figure.facecolor": SURF, "axes.facecolor": SURF, "axes.edgecolor": MUTED,
    "axes.labelcolor": INK2, "xtick.color": MUTED, "ytick.color": MUTED, "text.color": INK,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
    "grid.color": "#e6e5e0", "grid.linewidth": .6, "font.size": 9,
    "axes.titlesize": 10, "axes.titleweight": "bold"})
DIV = LinearSegmentedColormap.from_list("div", [ORANGE, "#f0efec", BLUE])

# Listas de variáveis usadas ao longo do notebook
NUM = ["taxa_alfabetizacao_2023", "media_portugues_2023", "nivel_2023", "meta_alfabetizacao_2024",
       "area_km2", "pop", "pib", "matematica_saeb_2023", "portugues_saeb_2023", "ideb_2023",
       "log_pop", "log_pib", "log_area_km2", "diff_meta_2024", "atingiu_meta_2024"]
CONT = [c for c in NUM if c not in ("nivel_2023", "atingiu_meta_2024")]

### Funções auxiliares (Spark)

Estas funções concentram os cálculos que se repetem. Todas recebem um `DataFrame` Spark e devolvem só números agregados.

In [0]:
def hist_spark(sdf, col, bins=50, lo=None, hi=None):
    # Histograma calculado no Spark: atribui cada linha a um bin e conta por bin.
    if lo is None or hi is None:
        lo, hi = sdf.agg(F.min(col), F.max(col)).first()
    w = (hi - lo) / bins
    cont = (sdf.select(F.least(F.floor((F.col(col) - lo) / w), F.lit(bins - 1)).cast("int").alias("b"))
              .groupBy("b").count().toPandas().set_index("b")["count"]
              .reindex(range(bins), fill_value=0))
    return lo + w * np.arange(bins), cont.values, w


def quantis_spark(sdf, col, ps):
    # Quantis exatos (interpolação linear, igual ao pandas).
    return sdf.agg(F.percentile(col, ps)).first()[0]


def vetor(sdf, cols, out="v"):
    # Monta a coluna vetorial exigida pelo pyspark.ml.
    base = sdf.select([F.col(c).cast("double").alias(c) for c in cols])
    return VectorAssembler(inputCols=cols, outputCol=out).transform(base).select(out)

## 1. Estrutura e qualidade da base
Valores nulos, duplicatas, cardinalidade e cobertura das UFs.

In [0]:
nulos = df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).toPandas().T
nulos.columns = ["nulos"]
cardinalidade = df.agg(*[F.countDistinct(c).alias(c) for c in df.columns]).toPandas().T
cardinalidade.columns = ["n_unicos"]
tipos = pd.Series(dict(df.dtypes), name="tipo")
pd.concat([tipos, nulos, cardinalidade], axis=1)

In [0]:
dup_linhas = N - df.dropDuplicates().count()
dup_id = N - df.select("id_municipio").distinct().count()
nomes_repetidos = N - df.select("nome_municipio").distinct().count()

TODAS_UFS = set("AC AL AM AP BA CE DF ES GO MA MG MS MT PA PB PE PI PR RJ RN RO RR RS SC SE SP TO".split())
ufs_base = {r[0] for r in df.select("sigla_uf").distinct().collect()}

print("Linhas duplicadas:          ", dup_linhas)
print("id_municipio duplicado:     ", dup_id)
print("Nomes de município repetidos:", nomes_repetidos, "(homônimos em UFs diferentes)")
print("UFs presentes:", len(ufs_base), "| ausentes:", sorted(TODAS_UFS - ufs_base))
print("Exemplos de nomes (encoding):", [r[0] for r in df.select("nome_municipio").limit(3).collect()])

## 2. Variável resposta e vazamento
Distribuição do `ALVO`, relação com `Y` e identidades entre colunas derivadas.

In [0]:
df.groupBy("ALVO").agg(
    F.count("*").alias("n"),
    F.round(F.count("*") / N * 100, 2).alias("pct"),
    F.min("Y").alias("Y_min"), F.max("Y").alias("Y_max"), F.round(F.avg("Y"), 2).alias("Y_medio")
).orderBy("ALVO").toPandas()

In [0]:
checagens = df.agg(
    # ALVO é exatamente 1{Y >= 743}?
    F.avg(((F.col("Y") >= 743).cast("int") == F.col("ALVO")).cast("int")).alias("concord_ALVO_Y743"),
    F.corr("Y", "media_portugues_2023").alias("corr_Y_mediaPort2023"),
    # diff_meta_2024 = taxa - meta ?
    F.max(F.abs(F.col("taxa_alfabetizacao_2023") - F.col("meta_alfabetizacao_2024") - F.col("diff_meta_2024"))).alias("max_erro_diff"),
    # atingiu_meta_2024 = 1{diff >= 0} ?
    F.avg(((F.col("diff_meta_2024") >= 0).cast("int") == F.col("atingiu_meta_2024")).cast("int")).alias("concord_atingiu"),
    # log_* = log1p(*) ?
    F.max(F.abs(F.log1p("pop") - F.col("log_pop"))).alias("erro_log1p_pop"),
    F.max(F.abs(F.log1p("pib") - F.col("log_pib"))).alias("erro_log1p_pib"),
    F.max(F.abs(F.log1p("area_km2") - F.col("log_area_km2"))).alias("erro_log1p_area"),
    F.countDistinct("meta_alfabetizacao_2030").alias("valores_meta_2030"),
).toPandas().T
checagens.columns = ["valor"]
checagens

In [0]:
print("Cortes de nivel_2023 (faixas de media_portugues_2023):")
display(df.groupBy("nivel_2023").agg(F.min("media_portugues_2023").alias("min"),
                                     F.max("media_portugues_2023").alias("max"),
                                     F.count("*").alias("n")).orderBy("nivel_2023").toPandas())

print("ALVO x (media_portugues_2023 >= 743):")
(df.withColumn("mp2023_ge_743", (F.col("media_portugues_2023") >= 743).cast("int"))
   .groupBy("ALVO").pivot("mp2023_ge_743", [0, 1]).count().orderBy("ALVO").toPandas())

In [0]:
vc = df.groupBy("ALVO").count().orderBy("ALVO").toPandas()
edges, _, w = hist_spark(df, "Y", bins=64, lo=680, hi=872)
hy = {k: hist_spark(df.filter(F.col("ALVO") == k), "Y", bins=64, lo=680, hi=872)[1] for k in (0, 1)}

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6), gridspec_kw={"width_ratios": [1, 2.2]})
ax[0].bar(vc.ALVO.astype(str), vc["count"], color=[ORANGE, BLUE], width=.55)
for i, v in enumerate(vc["count"]):
    ax[0].text(i, v + 50, f"{v} ({v / N:.1%})", ha="center", color=INK2)
ax[0].set_ylim(0, vc["count"].max() * 1.15); ax[0].set_title("Distribuição de ALVO", loc="left")
ax[0].set_ylabel("municípios"); ax[0].grid(axis="x", visible=False)
for k in (0, 1):
    ax[1].bar(edges, hy[k], width=w, align="edge", color=COR[k], edgecolor=SURF, linewidth=.5, label=f"ALVO = {k}")
ax[1].axvline(743, color=INK, ls="--", lw=1); ax[1].text(745, max(hy[1].max(), hy[0].max()) * .92, "Y = 743")
ax[1].set_title("Y por classe de ALVO: separação perfeita em 743", loc="left"); ax[1].set_xlabel("Y")
ax[1].legend(frameon=False); fig.tight_layout(); plt.show()

## 3. Estatísticas descritivas
Média, desvio-padrão, quartis exatos, assimetria e curtose calculados em uma única agregação Spark.

> O Spark fornece `skewness`/`kurtosis` **populacionais** (g₁, g₂). Para ficar igual ao pandas, aplicamos a correção amostral (G₁, G₂) de Joanes & Gill (1998).

In [0]:
COLS_DESC = ["Y"] + NUM
exprs = []
for c in COLS_DESC:
    exprs += [F.count(c).alias(f"{c}|count"), F.mean(c).alias(f"{c}|mean"), F.stddev_samp(c).alias(f"{c}|std"),
              F.min(c).alias(f"{c}|min"), F.percentile(c, 0.25).alias(f"{c}|25%"), F.percentile(c, 0.5).alias(f"{c}|50%"),
              F.percentile(c, 0.75).alias(f"{c}|75%"), F.max(c).alias(f"{c}|max"),
              F.skewness(c).alias(f"{c}|g1"), F.kurtosis(c).alias(f"{c}|g2"), F.countDistinct(c).alias(f"{c}|n_unicos")]
linha = df.agg(*exprs).first().asDict()

desc = pd.DataFrame({c: {k.split("|")[1]: v for k, v in linha.items() if k.split("|")[0] == c} for c in COLS_DESC}).T
n = desc["count"]
desc["assimetria"] = desc["g1"] * np.sqrt(n * (n - 1)) / (n - 2)                       # G1 (igual ao pandas)
desc["curtose"] = ((n + 1) * desc["g2"] + 6) * (n - 1) / ((n - 2) * (n - 3))          # G2 (igual ao pandas)
desc = desc.drop(columns=["g1", "g2"])
desc.round(3)

In [0]:
HV = ["taxa_alfabetizacao_2023", "media_portugues_2023", "meta_alfabetizacao_2024", "matematica_saeb_2023",
      "portugues_saeb_2023", "ideb_2023", "diff_meta_2024", "pop", "log_pop", "pib", "log_pib",
      "area_km2", "log_area_km2", "Y"]
fig, axs = plt.subplots(4, 4, figsize=(13, 10)); axs = axs.ravel()
for a, c in zip(axs, HV):
    e, h, w = hist_spark(df, c, bins=50)
    a.bar(e, h, width=w, align="edge", color=BLUE, edgecolor=SURF, linewidth=.4)
    a.set_title(f"{c}\nassimetria = {desc.loc[c, 'assimetria']:.2f}", loc="left", fontsize=8.5)
for a in axs[len(HV):]:
    a.axis("off")
fig.suptitle("Distribuições univariadas", x=.01, ha="left", fontweight="bold"); fig.tight_layout(); plt.show()

### Outliers pela regra de Tukey (1,5 × IIQ)

In [0]:
exprs = []
for c in CONT + ["Y"]:
    q1, q3 = desc.loc[c, "25%"], desc.loc[c, "75%"]
    iiq = q3 - q1
    exprs.append(F.sum(((F.col(c) < q1 - 1.5 * iiq) | (F.col(c) > q3 + 1.5 * iiq)).cast("int")).alias(c))
outliers = df.agg(*exprs).toPandas().T.rename(columns={0: "n_outliers_IIQ"})
outliers

In [0]:
extremos = df.agg(F.sum((F.col("taxa_alfabetizacao_2023") == 100).cast("int")).alias("taxa_alfabetizacao_100"),
                  F.sum((F.col("ideb_2023") >= 9).cast("int")).alias("ideb_maior_igual_9")).toPandas()
display(extremos)
print("Maiores populações:")
display(df.orderBy(F.desc("pop")).select("id_municipio", "nome_municipio", "sigla_uf", "pop").limit(5).toPandas())
print("Maiores Ideb:")
display(df.orderBy(F.desc("ideb_2023")).select("nome_municipio", "sigla_uf", "ideb_2023", "matematica_saeb_2023",
                                               "portugues_saeb_2023", "taxa_alfabetizacao_2023", "pop").limit(6).toPandas())
print("Menores Ideb:")
df.orderBy("ideb_2023").select("nome_municipio", "sigla_uf", "ideb_2023", "pop").limit(5).toPandas()

## 4. Matrizes de correlação
`pyspark.ml.stat.Correlation` calcula as matrizes de Pearson e Spearman sobre um vetor com todas as variáveis numéricas.

In [0]:
CC = ["ALVO", "Y"] + NUM
v = vetor(df, CC)
pearson = pd.DataFrame(Correlation.corr(v, "v", "pearson").first()[0].toArray(), index=CC, columns=CC)
spearman = pd.DataFrame(Correlation.corr(v, "v", "spearman").first()[0].toArray(), index=CC, columns=CC)


def heatmap(m, titulo):
    fig, ax = plt.subplots(figsize=(10, 8.5))
    im = ax.imshow(m.values, cmap=DIV, vmin=-1, vmax=1)
    ax.set_xticks(range(len(m))); ax.set_xticklabels(m.columns, rotation=60, ha="right")
    ax.set_yticks(range(len(m))); ax.set_yticklabels(m.index); ax.grid(False)
    for i in range(len(m)):
        for j in range(len(m)):
            val = m.values[i, j]
            ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=6.5, color="white" if abs(val) > .65 else INK)
    cb = fig.colorbar(im, ax=ax, shrink=.75); cb.outline.set_visible(False)
    ax.set_title(titulo, loc="left"); fig.tight_layout(); plt.show()


heatmap(spearman, "Matriz de correlação de Spearman (variáveis numéricas)")
heatmap(pearson, "Matriz de correlação de Pearson (variáveis numéricas)")

In [0]:
corr_alvo = (pd.DataFrame({"pearson": pearson["ALVO"], "spearman": spearman["ALVO"]})
               .drop("ALVO").sort_values("spearman", key=abs, ascending=False))
corr_alvo.round(3)

In [0]:
# Pares fortemente correlacionados (|rho de Spearman| >= 0,7)
pares = [(a, b, pearson.loc[a, b], spearman.loc[a, b])
         for i, a in enumerate(NUM) for b in NUM[i + 1:] if abs(spearman.loc[a, b]) >= .7]
pd.DataFrame(pares, columns=["var_1", "var_2", "pearson", "spearman"]).sort_values("spearman", key=abs, ascending=False).round(3)

### VIF: fator de inflação da variância
Para cada variável, ajustamos no Spark uma regressão linear dela contra as demais (`LinearRegression`) e calculamos VIF = 1 / (1 − R²).

> Quando há colinearidade **exata** (`diff = taxa − meta`), a matriz é singular. O Spark avisa e passa para um solver iterativo, então o VIF sai numericamente enorme ou infinito. O valor exato não importa: o que importa é a ordem de grandeza.

In [0]:
def vif_spark(sdf, cols):
    base = sdf.select([F.col(c).cast("double").alias(c) for c in cols]).cache()
    res = {}
    for c in cols:
        outras = [x for x in cols if x != c]
        dados = VectorAssembler(inputCols=outras, outputCol="X").transform(base)
        r2 = LinearRegression(featuresCol="X", labelCol=c, regParam=0.0).fit(dados).summary.r2
        res[c] = np.inf if r2 >= 1 else 1 / (1 - r2)
    base.unpersist()
    return pd.Series(res, name="VIF")


VIF_TODAS = ["taxa_alfabetizacao_2023", "media_portugues_2023", "meta_alfabetizacao_2024", "matematica_saeb_2023",
             "portugues_saeb_2023", "ideb_2023", "log_pop", "log_pib", "log_area_km2", "diff_meta_2024"]
VIF_REDUZ = ["media_portugues_2023", "portugues_saeb_2023", "matematica_saeb_2023", "ideb_2023",
             "log_pop", "log_pib", "log_area_km2"]
pd.concat([vif_spark(df, VIF_TODAS).rename("VIF (todas)"), vif_spark(df, VIF_REDUZ).rename("VIF (reduzido)")], axis=1)

## 5. Relação das variáveis numéricas com o ALVO

### 5.1 AUC univariada e teste de Mann-Whitney
- **AUC:** `BinaryClassificationEvaluator` usando a própria variável como escore. Com `numBins=0` não há reamostragem da curva, então o resultado é exato.
- **Mann-Whitney:** não existe pronto no Spark. Implementamos via postos médios com funções de janela: U = R₁ − n₁(n₁+1)/2, aproximação normal com correção de empates e de continuidade (a mesma do `scipy.stats.mannwhitneyu`).

> A janela `Window.orderBy` sem partição leva todos os dados para uma única partição. Com cerca de 5 mil linhas isso não é problema. Em bases muito grandes, prefira apenas a AUC, que é equivalente.

In [0]:
avaliador = BinaryClassificationEvaluator(labelCol="ALVO", rawPredictionCol="escore",
                                          metricName="areaUnderROC", numBins=0)


def auc_spark(sdf, col):
    d = sdf.select(F.col("ALVO").cast("double").alias("ALVO"), F.col(col).cast("double").alias("escore"))
    return avaliador.evaluate(d)


def mann_whitney_spark(sdf, col, label="ALVO"):
    d = sdf.select(F.col(col).cast("double").alias("x"), F.col(label).alias("g"))
    # posto médio em caso de empate: rank mínimo + (tamanho do grupo empatado - 1) / 2
    d = (d.withColumn("rk", F.rank().over(Window.orderBy("x")))
          .withColumn("t", F.count("*").over(Window.partitionBy("x")))
          .withColumn("posto", F.col("rk") + (F.col("t") - 1) / 2))
    g = {r["g"]: (r["n"], r["R"]) for r in d.groupBy("g").agg(F.count("*").alias("n"), F.sum("posto").alias("R")).collect()}
    (n1, R1), (n0, _) = g[1], g[0]
    empates = d.groupBy("x").count().agg(F.sum(F.pow("count", 3) - F.col("count"))).first()[0]
    Ntot = n0 + n1
    U1 = R1 - n1 * (n1 + 1) / 2
    mu = n1 * n0 / 2
    sigma = np.sqrt(n1 * n0 / 12 * ((Ntot + 1) - empates / (Ntot * (Ntot - 1))))
    z = (U1 - mu - 0.5 * np.sign(U1 - mu)) / sigma
    return U1, 2 * stats.norm.sf(abs(z)), U1 / (n1 * n0)


# médias e medianas por classe em uma agregação
por_classe = (df.groupBy("ALVO")
                .agg(*[F.mean(c).alias(f"{c}|media") for c in NUM], *[F.percentile(c, .5).alias(f"{c}|mediana") for c in NUM])
                .toPandas().set_index("ALVO"))

linhas = []
for c in NUM:
    auc = auc_spark(df, c)
    U, p, auc_mw = mann_whitney_spark(df, c)
    linhas.append({"variavel": c,
                   "mediana_0": por_classe.loc[0, f"{c}|mediana"], "mediana_1": por_classe.loc[1, f"{c}|mediana"],
                   "media_0": por_classe.loc[0, f"{c}|media"], "media_1": por_classe.loc[1, f"{c}|media"],
                   "AUC": max(auc, 1 - auc), "sentido": "+" if auc >= .5 else "-",
                   "AUC_via_U": max(auc_mw, 1 - auc_mw), "U": U, "p_mann_whitney": p})
sep = pd.DataFrame(linhas).set_index("variavel").sort_values("AUC", ascending=False)
sep.style.format({"p_mann_whitney": "{:.2e}", "U": "{:,.0f}"}, precision=3)

In [0]:
s = sep["AUC"].sort_values()
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(s.index, s.values - .5, left=.5, color=BLUE, height=.6)
for i, (var, val) in enumerate(s.items()):
    ax.text(val + .005, i, f"{val:.3f} ({sep.loc[var, 'sentido']})", va="center", fontsize=8, color=INK2)
ax.set_xlim(.5, 1.05); ax.grid(axis="y", visible=False)
ax.set_title("Poder discriminante univariado (AUC) em relação a ALVO", loc="left"); fig.tight_layout(); plt.show()

### 5.2 Densidades por classe (KDE)
`pyspark.mllib.stat.KernelDensity` com kernel gaussiano e largura de banda pela regra de Scott (h = s · n^(−1/5)), a mesma usada pelo `scipy.stats.gaussian_kde`.

In [0]:
DV = ["media_portugues_2023", "taxa_alfabetizacao_2023", "portugues_saeb_2023", "matematica_saeb_2023",
      "ideb_2023", "meta_alfabetizacao_2024", "diff_meta_2024", "log_pop", "log_pib", "log_area_km2"]


def kde_spark(sdf, col, grade):
    n, s = sdf.agg(F.count(col), F.stddev_samp(col)).first()
    kd = KernelDensity()
    kd.setSample(sdf.select(col).rdd.map(lambda r: float(r[0])))
    kd.setBandwidth(s * n ** (-1 / 5))
    return np.array(kd.estimate(list(map(float, grade))))


fig, axs = plt.subplots(2, 5, figsize=(15, 6)); axs = axs.ravel()
for a, c in zip(axs, DV):
    lo, hi = quantis_spark(df, c, [.005, .995])
    xs = np.linspace(lo, hi, 300)
    for k in (0, 1):
        y = kde_spark(df.filter(F.col("ALVO") == k), c, xs)
        a.fill_between(xs, y, color=COR[k], alpha=.18, lw=0)
        a.plot(xs, y, color=COR[k], lw=2, label=f"ALVO = {k}")
    a.set_title(f"{c}\nAUC univariada = {sep.loc[c, 'AUC']:.3f}", loc="left", fontsize=8.5); a.set_yticks([])
axs[0].legend(frameon=False, fontsize=8)
fig.suptitle("Densidade das variáveis por classe de ALVO", x=.01, ha="left", fontweight="bold"); fig.tight_layout(); plt.show()

### 5.3 Boxplots por classe
Quartis, bigodes (último ponto dentro de 1,5·IIQ) e outliers calculados no Spark. O `matplotlib` só desenha, via `ax.bxp`.

In [0]:
def box_stats_spark(sdf, col, label="ALVO"):
    caixas = []
    for k in (0, 1):
        g = sdf.filter(F.col(label) == k).select(col)
        q1, med, q3 = quantis_spark(g, col, [.25, .5, .75])
        lim_inf, lim_sup = q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1)
        b_inf, b_sup = g.filter(F.col(col).between(lim_inf, lim_sup)).agg(F.min(col), F.max(col)).first()
        fliers = [r[0] for r in g.filter((F.col(col) < lim_inf) | (F.col(col) > lim_sup)).collect()]
        caixas.append(dict(label=f"ALVO {k}", q1=q1, med=med, q3=q3, whislo=b_inf, whishi=b_sup, fliers=fliers))
    return caixas


fig, axs = plt.subplots(2, 5, figsize=(15, 6)); axs = axs.ravel()
for a, c in zip(axs, DV):
    bp = a.bxp(box_stats_spark(df, c), widths=.5, patch_artist=True,
               flierprops=dict(marker="o", markersize=2, alpha=.3, markeredgecolor=MUTED),
               medianprops=dict(color=INK, lw=1.5))
    for caixa, k in zip(bp["boxes"], (0, 1)):
        caixa.set_facecolor(COR[k]); caixa.set_alpha(.75); caixa.set_edgecolor(COR[k])
    a.set_title(c, loc="left", fontsize=8.5)
fig.suptitle("Boxplots por classe de ALVO", x=.01, ha="left", fontweight="bold"); fig.tight_layout(); plt.show()

### 5.4 Relações bivariadas
Uma amostra de cerca de 2.500 municípios é levada ao driver apenas para desenhar os pontos. As correlações exibidas vêm da matriz Spark completa.

In [0]:
amostra = df.sample(fraction=2500 / N, seed=1).select("ALVO", "Y", "media_portugues_2023", "taxa_alfabetizacao_2023",
                                                       "matematica_saeb_2023", "portugues_saeb_2023").toPandas()
PARES = [("media_portugues_2023", "Y"), ("media_portugues_2023", "taxa_alfabetizacao_2023"),
         ("matematica_saeb_2023", "portugues_saeb_2023")]
fig, axs = plt.subplots(1, 3, figsize=(14, 4.2))
for a, (x, y) in zip(axs, PARES):
    for k in (0, 1):
        s_ = amostra[amostra.ALVO == k]
        a.scatter(s_[x], s_[y], s=9, color=COR[k], alpha=.55, lw=0, label=f"ALVO = {k}")
    a.set_xlabel(x); a.set_ylabel(y); a.set_title(f"ρ Spearman = {spearman.loc[x, y]:.2f}", loc="left")
axs[0].axhline(743, color=INK, ls="--", lw=.8); axs[0].axvline(743, color=INK, ls="--", lw=.8); axs[0].legend(frameon=False)
fig.suptitle(f"Relações bivariadas (amostra de {len(amostra)} municípios)", x=.01, ha="left", fontweight="bold")
fig.tight_layout(); plt.show()

## 6. Variáveis categóricas e discretas

### 6.1 Qui-quadrado e V de Cramér
`pyspark.ml.stat.ChiSquareTest` testa a independência entre cada variável categórica (indexada com `StringIndexer`) e o `ALVO`. O V de Cramér é calculado a partir do χ².

> O Spark **não** aplica a correção de continuidade de Yates em tabelas 2×2. Por isso o χ² de `atingiu_meta_2024` fica um pouco acima do valor do `scipy` (327,1). O V de Cramér praticamente não muda.

In [0]:
def chi2_cramer_spark(sdf, col, label="ALVO"):
    if dict(sdf.dtypes)[col] == "string":
        sdf = StringIndexer(inputCol=col, outputCol="cat").fit(sdf).transform(sdf)
    else:
        sdf = sdf.withColumn("cat", F.col(col).cast("double"))
    d = VectorAssembler(inputCols=["cat"], outputCol="f").transform(sdf).select("f", F.col(label).cast("double").alias("y"))
    r = ChiSquareTest.test(d, "f", "y").first()
    chi2, p, gl = float(r.statistics[0]), float(r.pValues[0]), int(r.degreesOfFreedom[0])
    k = sdf.select(col).distinct().count()
    v = np.sqrt(chi2 / (N * (min(k, 2) - 1)))
    return {"chi2": chi2, "gl": gl, "p": p, "V_cramer": v}


pd.DataFrame({c: chi2_cramer_spark(df, c) for c in ["sigla_uf", "nivel_2023", "nome_regiao", "atingiu_meta_2024"]}).T \
  .style.format({"chi2": "{:,.1f}", "gl": "{:.0f}", "p": "{:.2e}", "V_cramer": "{:.3f}"})

### 6.2 Taxa de `ALVO = 1` por grupo

In [0]:
def taxa_por(sdf, col):
    return (sdf.groupBy(col).agg(F.count("*").alias("n"), F.avg("ALVO").alias("pct_alvo1"))
               .orderBy("pct_alvo1").toPandas().set_index(col))

rg, uf, nv = taxa_por(df, "nome_regiao"), taxa_por(df, "sigla_uf"), taxa_por(df, "nivel_2023").sort_index()
media_geral = df.agg(F.avg("ALVO")).first()[0]
display(rg); display(nv); display(taxa_por(df, "atingiu_meta_2024"))
uf.T

In [0]:
fig, ax = plt.subplots(1, 2, figsize=(13, 5), gridspec_kw={"width_ratios": [1, 2]})
ax[0].barh(rg.index, rg.pct_alvo1, color=BLUE, height=.55)
for i, (m, n_) in enumerate(zip(rg.pct_alvo1, rg.n)):
    ax[0].text(m + .01, i, f"{m:.1%}  (n={n_})", va="center", color=INK2, fontsize=8)
ax[0].axvline(media_geral, color=INK, ls="--", lw=1); ax[0].set_xlim(0, 1.2)
ax[0].set_title("% ALVO = 1 por região", loc="left"); ax[0].grid(axis="y", visible=False)
ax[1].bar(uf.index, uf.pct_alvo1, color=BLUE, width=.65)
ax[1].axhline(media_geral, color=INK, ls="--", lw=1)
ax[1].text(-.4, media_geral + .02, f"média geral {media_geral:.1%}", fontsize=8)
ax[1].set_title("% ALVO = 1 por UF (ordenado)", loc="left"); ax[1].set_ylim(0, 1.05); ax[1].grid(axis="x", visible=False)
fig.tight_layout(); plt.show()

### 6.3 População e PIB per capita
Os decis e quintis são criados com `F.ntile` sobre uma janela ordenada, o que gera grupos de mesmo tamanho. Pode haver pequenas diferenças em relação ao `pd.qcut` quando há empates exatamente no limite entre faixas.

In [0]:
df_q = (df.withColumn("pib_per_capita", F.col("pib") / F.col("pop"))
          .withColumn("decil_pop", F.ntile(10).over(Window.orderBy("pop")))
          .withColumn("quintil_pibpc", F.ntile(5).over(Window.orderBy("pib_per_capita"))))

pop_decil = (df_q.groupBy("decil_pop").agg(F.min("pop").alias("pop_min"), F.max("pop").alias("pop_max"),
                                           F.avg("ALVO").alias("pct_alvo1")).orderBy("decil_pop").toPandas())
pibpc_quintil = (df_q.groupBy("quintil_pibpc").agg(F.min("pib_per_capita").alias("min"), F.max("pib_per_capita").alias("max"),
                                                   F.avg("ALVO").alias("pct_alvo1")).orderBy("quintil_pibpc").toPandas())
auc_pibpc = auc_spark(df_q, "pib_per_capita")
rho_pibpc = Correlation.corr(vetor(df_q, ["pib_per_capita", "ALVO"]), "v", "spearman").first()[0].toArray()[0, 1]
display(pop_decil); display(pibpc_quintil)
print(f"PIB per capita: AUC = {auc_pibpc:.3f} | rho Spearman com ALVO = {rho_pibpc:.3f}")

In [0]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
ax[0].bar(nv.index.astype(str), nv.pct_alvo1, color=BLUE, width=.6)
for i, (m, n_) in enumerate(zip(nv.pct_alvo1, nv.n)):
    ax[0].text(i, m + .02, f"{m:.0%}\nn={n_}", ha="center", fontsize=7.5, color=INK2)
ax[0].set_ylim(0, 1.25); ax[0].set_xlabel("nivel_2023"); ax[0].grid(axis="x", visible=False)
ax[0].set_title("% ALVO = 1 por nivel_2023", loc="left")
ax[1].plot("D" + pop_decil.decil_pop.astype(str), pop_decil.pct_alvo1, color=BLUE, lw=2, marker="o",
           markersize=7, markeredgecolor=SURF, markeredgewidth=2)
ax[1].set_ylim(.4, .9); ax[1].set_xlabel("decil de pop (D1 = menores)")
ax[1].set_title("% ALVO = 1 por decil de população", loc="left")
fig.tight_layout(); plt.show()

## 7. Baselines
São as regras triviais que qualquer modelo supervisionado precisa superar.

In [0]:
baselines = df.agg(
    F.avg((F.col("ALVO") == 1).cast("int")).alias("acuracia_sempre_1"),
    F.avg(((F.col("media_portugues_2023") >= 743).cast("int") == F.col("ALVO")).cast("int")).alias("acuracia_regra_2023_ge_743"),
).toPandas().T.rename(columns={0: "acuracia"})
baselines

## Conclusões
1. **`Y` é vazamento:** `ALVO = 1{Y ≥ 743}`. Não use `Y` como feature.
2. A base está limpa: sem nulos e sem duplicatas. Faltam DF e RR.
3. O alvo é 66% de 1 e 34% de 0.
4. O melhor preditor isolado é `media_portugues_2023` (AUC ≈ 0,84). O baseline da regra de 2023 é ≈ 78% de acurácia.
5. Há colinearidade exata (`diff = taxa − meta`) e três blocos correlacionados: alfabetização 2023, Saeb/Ideb e porte.
6. A UF tem associação forte com o alvo (V ≈ 0,62). Municípios menores e com maior PIB per capita atingem o alvo com mais frequência.

In [0]:
spark.stop()